# ASL Index & Climatology Workflow

This notebook drives the ASL (Amundsen Sea Low) analysis workflow, which includes:
1. Identifying ASL locations and generating indices from PSL data.
2. Generating seasonal and annual cycle time series (climatology step 1).
3. Calculating mean climatology and bias against observation/reanalysis (climatology step 2).
4. Performing ASL lead-lag regression and correlation analysis.

In [23]:
import os
import sys
import glob
import collections
import re
import importlib

def configure_proj_data():
    candidates = [
        os.environ.get("PROJ_DATA"),
        os.environ.get("PROJ_LIB"),
        os.path.join(sys.prefix, "share", "proj"),
    ]

    for path in candidates:
        if path and os.path.exists(os.path.join(path, "proj.db")):
            os.environ["PROJ_DATA"] = path
            os.environ["PROJ_LIB"] = path
            return path

    return None


PROJ_DATA = configure_proj_data()

# Ensure our shared libraries are discoverable
sys.path.append("..")
import util.common as common_utils
import util.asl as asl_utils
import util.asl_workflow as asl_workflow

common_utils = importlib.reload(common_utils)
asl_utils = importlib.reload(asl_utils)
asl_workflow = importlib.reload(asl_workflow)

Case = common_utils.Case
run_asl_index_generation = asl_utils.run_asl_index_generation
run_asl_leadlag_analysis = asl_utils.run_asl_leadlag_analysis
run_asl_index_clim_step1 = asl_utils.run_asl_index_clim_step1
run_asl_index_clim_step2 = asl_utils.run_asl_index_clim_step2

print(f"Reloaded util.common from {common_utils.__file__}")
print(f"Reloaded util.asl from {asl_utils.__file__}")
print(f"Reloaded util.asl_workflow from {asl_workflow.__file__}")

Reloaded util.common from /lcrc/group/e3sm2/ac.szhang/E3SMv21_testings/v21_sorrm_polar_review/polar_analysis/jupyter/../util/common.py
Reloaded util.asl from /lcrc/group/e3sm2/ac.szhang/E3SMv21_testings/v21_sorrm_polar_review/polar_analysis/jupyter/../util/asl.py
Reloaded util.asl_workflow from /lcrc/group/e3sm2/ac.szhang/E3SMv21_testings/v21_sorrm_polar_review/polar_analysis/jupyter/../util/asl_workflow.py


In [24]:
# Initialize Dask Client for Diagnostics Dashboard (Optional)
from dask.distributed import Client

def set_worker_proj_data(proj_data):
    import os

    os.environ["PROJ_DATA"] = proj_data
    os.environ["PROJ_LIB"] = proj_data
    return os.path.exists(os.path.join(proj_data, "proj.db"))


client = Client()

if PROJ_DATA:
    worker_proj_ready = client.run(set_worker_proj_data, PROJ_DATA)
    print(f"Configured PROJ_DATA={PROJ_DATA}")
    print(f"Workers with proj.db: {sum(worker_proj_ready.values())}/{len(worker_proj_ready)}")
else:
    print("WARNING: Could not locate proj.db; Cartopy/GDAL may emit PROJ errors.")

client

/lcrc/soft/climate/e3sm-unified/e3smu_1_12_0/chrysalis/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41365 instead
  warnings.warn(


Configured PROJ_DATA=/lcrc/soft/climate/e3sm-unified/e3smu_1_12_0/chrysalis/conda/envs/e3sm_unified_1.12.0_login/share/proj
Workers with proj.db: 16/16


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:41365/status,
Dashboard: http://127.0.0.1:41365/status,Workers: 16
Total threads: 128,Total memory: 0.98 TiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38507,Workers: 0
Dashboard: http://127.0.0.1:41365/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:37481,Total threads: 8
Dashboard: http://127.0.0.1:33999/status,Memory: 62.95 GiB
Nanny: tcp://127.0.0.1:41295,


In [25]:
# ============================================================
# Global configuration paths
# ============================================================
top_path = "/lcrc/group/e3sm2/ac.szhang/E3SMv21_testings/v3_polar_paper"
run_path = "/lcrc/group/e3sm/ac.szhang/acme_scratch/data/pcmdi/model/monthly"
run_mask = "/lcrc/group/e3sm/ac.szhang/acme_scratch/data/pcmdi/model/fixed/sftlf"
model_root = "/lcrc/group/e3sm2/ac.szhang/E3SMv21_testings"

# ============================================================
# Shared output-layout definitions
#
# These describe the stable parts of the output directory.
# The final chunk directory, such as 5yr, 10yr, or 30yr, is
# configured separately for each model simulation.
# ============================================================
DATA_LAYOUTS = {
    "atm_ts": {
        "component": "atm",
        "grid": "180x360_aave",
        "product": "ts",
        "frequency": "monthly",
    },
    "atm_clim": {
        "component": "atm",
        "grid": "180x360_aave",
        "product": "clim",
        "frequency": None,
    },
    "lnd_ts": {
        "component": "lnd",
        "grid": "180x360_aave",
        "product": "ts",
        "frequency": "monthly",
    },
    "lnd_clim": {
        "component": "lnd",
        "grid": "180x360_aave",
        "product": "clim",
        "frequency": None,
    },
}


# ============================================================
# Model-case configurations
#
# Set "chunk" to:
#
#   "auto"  -> discover an available chunk directory
#   "5yr"   -> explicitly use the 5yr directory
#   "10yr"  -> explicitly use the 10yr directory
#   "30yr"  -> explicitly use the 30yr directory
#   None    -> use the product directory without a chunk
#
# Historical and SSP370 data from the SORRM histssp370 cases
# are distinguished by the requested analysis period.
# ============================================================

MODEL_CASES = {
    # ========================================================
    # E3SM v2.1 LR historical ensemble
    # ========================================================

    "v2.1-LR-HIST-0101": {
        "model_name": "v2_1.LR.historical_0101",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0101",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2.1-LR-HIST-0151": {
        "model_name": "v2_1.LR.historical_0151",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0151",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2.1-LR-HIST-0201": {
        "model_name": "v2_1.LR.historical_0201",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0201",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2.1-LR-HIST-0251": {
        "model_name": "v2_1.LR.historical_0251",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0251",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2.1-LR-HIST-0301": {
        "model_name": "v2_1.LR.historical_0301",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0301",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2.1-LR-PiControl": {
        "model_name": "v2_1.LR.piControl",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "piControl",
        "member": None,
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "10yr"},
            "atm_clim": {"chunk": "50yr"},
            "lnd_ts": {"chunk": "10yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    # ========================================================
    # E3SM v2.1 SORRM
    # ========================================================

    "v2.1-SORRM-Control": {
        "model_name": "v2_1.SORRM.control",
        "case_name": "v2_1-SORRM",
        "model_version": "v2.1",
        "configuration": "SORRM",
        "experiment": "piControl",
        "member": None,
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "50yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "50yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2.1-SORRM-HIST-0701": {
        "model_name": "v2_1.SORRM.histssp370_0701",
        "case_name": "v2_1-SORRM",
        "model_version": "v2.1",
        "configuration": "SORRM",
        "experiment": "historical+ssp370",
        "member": "0701",
        "variant": "standard",
        "outputs": {
             "atm_ts": {"chunk": "10yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "10yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2.1-SORRM-HIST-0751": {
        "model_name": "v2_1.SORRM.histssp370_0751",
        "case_name": "v2_1-SORRM",
        "model_version": "v2.1",
        "configuration": "SORRM",
        "experiment": "historical+ssp370",
        "member": "0751",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "10yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "10yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2.1-SORRM-HIST-0801": {
        "model_name": "v2_1.SORRM.histssp370_0801",
        "case_name": "v2_1-SORRM",
        "model_version": "v2.1",
        "configuration": "SORRM",
        "experiment": "historical+ssp370",
        "member": "0801",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "10yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "10yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2.1-SORRM-HIST-0701-FISMF": {
        "model_name": "v2_1.SORRM.histssp370-fismf_0701",
        "case_name": "v2_1-SORRM-fismf",
        "model_version": "v2.1",
        "configuration": "SORRM",
        "experiment": "historical+ssp370",
        "member": "0701",
        "variant": "fismf",
        "outputs": {
            "atm_ts": {"chunk": "10yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "10yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    # ========================================================
    # E3SM v3 LR AMIP ensemble
    # ========================================================

    "v3-LR-AMIP-0101": {
        "model_name": "v3.LR.amip_0101",
        "case_name": "v3-LR-AMIP",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "amip",
        "member": "0101",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3-LR-AMIP-0151": {
        "model_name": "v3.LR.amip_0151",
        "case_name": "v3-LR-AMIP",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "amip",
        "member": "0151",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3-LR-AMIP-0201": {
        "model_name": "v3.LR.amip_0201",
        "case_name": "v3-LR-AMIP",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "amip",
        "member": "0201",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    # ========================================================
    # E3SM v3 LR historical ensemble
    # ========================================================

    "v3-LR-HIST-0051": {
        "model_name": "v3.LR.historical_0051",
        "case_name": "v3-LR",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0051",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3-LR-HIST-0101": {
        "model_name": "v3.LR.historical_0101",
        "case_name": "v3-LR",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0101",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3-LR-HIST-0101-bcdt15m": {
        "model_name": "v3.LR.historical_0101_bcdt15m",
        "case_name": "v3-LR-bcdt15m",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0101",
        "variant": "bcdt15m",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3-LR-HIST-0151": {
        "model_name": "v3.LR.historical_0151",
        "case_name": "v3-LR",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0151",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3-LR-HIST-0201": {
        "model_name": "v3.LR.historical_0201",
        "case_name": "v3-LR",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0201",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "30yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    # ========================================================
    # E3SM v3 LR piControl
    # ========================================================

    "v3-LR-PiControl": {
        "model_name": "v3.LR.piControl",
        "case_name": "v3-LR",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "piControl",
        "member": None,
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "10yr"},
            "atm_clim": {"chunk": "100yr"},
            "lnd_ts": {"chunk": "10yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3-LR-PiControl-dismf": {
        "model_name": "v3.LR.piControl-scaled-dismf",
        "case_name": "v3-LR-scaled-dismf",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "piControl",
        "member": None,
        "variant": "scaled-dismf",
        "outputs": {
            "atm_ts": {"chunk": "5yr"},
            "atm_clim": {"chunk": "50yr"},
            "lnd_ts": {"chunk": "5yr"},
            "lnd_clim": {"chunk": "auto"},
        },
    },
}


# ============================================================
# Analysis periods
#
# The period is separated from MODEL_CASES because multiple
# analysis periods can be selected from the same simulation.
# For example, the SORRM histssp370 simulations contain both
# historical and SSP370 data.
# ============================================================

ANALYSIS_PERIODS = {
    "historical": {
        "start_ym": 195101,
        "end_ym": 201412,
    },
    "ssp370": {
        "start_ym": 201501,
        "end_ym": 210012,
    },
    "ssp245": {
        "start_ym": 201501,
        "end_ym": 205012,
    },
    "historical_ssp370": {
        "start_ym": 195101,
        "end_ym": 210012,
    },
    "historical_1985_2014": {
        "start_ym": 198501,
        "end_ym": 201412,
    },
    "future_2071_2100": {
        "start_ym": 207101,
        "end_ym": 210012,
    },
    "sorrm_piControl": {
        "start_ym": 80101,
        "end_ym": 100012,
    },
    "lr_piControl": {
        "start_ym": 101,
        "end_ym": 25012,
    },
    "v3_lr_piControl": {
        "start_ym": 101,
        "end_ym": 50012,
    },
    "v3_lr_piControl_spinup": {
        "start_ym": 101,
        "end_ym": 25012,
    },
    "amip": {
        "start_ym": 187101,
        "end_ym": 201412,
    },
    "cmip6_amip": {
        "start_ym": 197901,
        "end_ym": 201412,
    },
}


# ============================================================
# Filename parsing
#
# Supports filenames such as:
#
#   PSL_195101_196012.nc
#   PSL_080101_085012.nc
# ============================================================

_MODEL_FILE_PATTERN = re.compile(
    r"^(?P<variable>.+?)_"
    r"(?P<start>\d{5,6})_"
    r"(?P<end>\d{5,6})\.nc$"
)


def parse_model_filename(path):
    """
    Parse the variable and date range from a model-output filename.

    Returns
    -------
    dict or None
        Dictionary containing path, variable, start_ym, and end_ym.
        Returns None when the filename does not match the expected form.
    """
    basename = os.path.basename(path)
    match = _MODEL_FILE_PATTERN.match(basename)

    if match is None:
        return None

    return {
        "path": path,
        "variable": match.group("variable"),
        "start_ym": int(match.group("start")),
        "end_ym": int(match.group("end")),
    }


def split_ym(ym):
    """Split an integer YYYYMM value into year and month."""
    year, month = divmod(int(ym), 100)

    if month < 1 or month > 12:
        raise ValueError(f"Invalid year-month value: {ym}")

    return year, month


def month_index(ym):
    """Convert YYYYMM to a monotonically increasing month index."""
    year, month = split_ym(ym)
    return year * 12 + month - 1


# ============================================================
# Configuration helpers
# ============================================================

def get_output_config(case_key, output_key):
    """
    Merge the shared layout definition with the case-specific
    output configuration.
    """
    if case_key not in MODEL_CASES:
        valid = ", ".join(sorted(MODEL_CASES))
        raise KeyError(
            f"Unknown model case {case_key!r}. "
            f"Valid cases: {valid}"
        )

    if output_key not in DATA_LAYOUTS:
        valid = ", ".join(sorted(DATA_LAYOUTS))
        raise KeyError(
            f"Unknown output layout {output_key!r}. "
            f"Valid layouts: {valid}"
        )

    case = MODEL_CASES[case_key]
    outputs = case.get("outputs", {})

    if output_key not in outputs:
        valid = ", ".join(sorted(outputs))
        raise KeyError(
            f"Output {output_key!r} is not configured for "
            f"{case_key!r}. Available outputs: {valid}"
        )

    return {
        **DATA_LAYOUTS[output_key],
        **outputs[output_key],
    }


def get_output_base_path(case_key, output_key):
    """
    Construct the output directory without the final chunk directory.
    """
    case = MODEL_CASES[case_key]
    config = get_output_config(case_key, output_key)

    parts = [
        model_root,
        case["model_name"],
        "post",
        config["component"],
        config["grid"],
        config["product"],
    ]

    if config.get("frequency"):
        parts.append(config["frequency"])

    return os.path.join(*parts)


def discover_chunk_directories(base_path):
    """
    Return all available subdirectories under an output product path.

    Directories are sorted by their numeric year length when their
    names follow forms such as 5yr, 10yr, 30yr, or 50yr.
    """
    if not os.path.isdir(base_path):
        return []

    subdirectories = [
        path
        for path in glob.glob(os.path.join(base_path, "*"))
        if os.path.isdir(path)
    ]

    def chunk_sort_key(path):
        name = os.path.basename(path)
        match = re.fullmatch(r"(\d+)yr", name)

        if match:
            return 0, -int(match.group(1))

        return 1, name

    return sorted(subdirectories, key=chunk_sort_key)


def get_model_output_paths(case_key, output_key):
    """
    Return one or more configured output directories.

    When chunk="auto", all available chunk directories are returned.
    Searching all directories allows the date-coverage logic to choose
    the appropriate non-overlapping files.
    """
    config = get_output_config(case_key, output_key)
    base_path = get_output_base_path(case_key, output_key)
    chunk = config.get("chunk")

    if chunk is None:
        return [base_path]

    if chunk == "auto":
        directories = discover_chunk_directories(base_path)

        if not directories:
            raise FileNotFoundError(
                f"No chunk directories found under:\n{base_path}"
            )

        return directories

    output_path = os.path.join(base_path, chunk)

    if not os.path.isdir(output_path):
        raise FileNotFoundError(
            f"Configured output directory does not exist:\n"
            f"{output_path}"
        )

    return [output_path]


def identify_analysis_experiment(case_key, start_ym, end_ym):
    """
    Identify the forcing period represented by the selected dates.

    Combined SORRM historical+SSP370 simulations are classified from
    the selected time range.
    """
    if case_key not in MODEL_CASES:
        valid = ", ".join(sorted(MODEL_CASES))
        raise KeyError(
            f"Unknown model case {case_key!r}. "
            f"Valid cases: {valid}"
        )

    case_experiment = MODEL_CASES[case_key]["experiment"]

    if case_experiment != "historical+ssp370":
        return case_experiment

    if end_ym <= 201412:
        return "historical"

    if start_ym >= 201501:
        return "ssp370"

    return "historical+ssp370"


# ============================================================
# File selection
# ============================================================

def select_continuous_files(candidates, start_ym, end_ym):
    """
    Select a continuous, non-overlapping set of files.

    At each step, the file that covers the next required month and
    extends furthest forward is selected. This avoids simultaneously
    using overlapping 5yr, 10yr, 30yr, or 50yr files.
    """
    candidates = list(candidates)
    selected = []

    current_index = month_index(start_ym)
    final_index = month_index(end_ym)

    while current_index <= final_index:
        available = [
            item
            for item in candidates
            if (
                month_index(item["start_ym"])
                <= current_index
                <= month_index(item["end_ym"])
            )
        ]

        if not available:
            missing_year = current_index // 12
            missing_month = current_index % 12 + 1

            raise RuntimeError(
                "Gap detected in model-output coverage at "
                f"{missing_year:04d}-{missing_month:02d}"
            )

        best = max(
            available,
            key=lambda item: (
                month_index(item["end_ym"]),
                -month_index(item["start_ym"]),
            ),
        )

        selected.append(best)
        current_index = month_index(best["end_ym"]) + 1

    unique = []
    seen_paths = set()

    for item in selected:
        if item["path"] not in seen_paths:
            unique.append(item)
            seen_paths.add(item["path"])

    return unique


def get_model_files(
    case_key,
    output_key="atm_ts",
    variable="PSL",
    start_ym=None,
    end_ym=None,
):
    """
    Return sorted, continuous, non-overlapping model files.

    Parameters
    ----------
    case_key : str
        Key in MODEL_CASES.

    output_key : str
        Output type, such as "atm_ts", "atm_clim",
        "lnd_ts", or "lnd_clim".

    variable : str
        Variable filename prefix, such as "PSL", "TREFHT",
        or "LANDFRAC".

    start_ym, end_ym : int, optional
        Requested time range in YYYYMM format.

        When omitted, all matching files are returned. When supplied,
        only a continuous set covering the requested period is returned.

    Returns
    -------
    list[str]
        Selected model-output file paths.
    """
    output_paths = get_model_output_paths(
        case_key=case_key,
        output_key=output_key,
    )

    files = []

    for output_path in output_paths:
        pattern = os.path.join(output_path, f"{variable}_*.nc")
        files.extend(glob.glob(pattern))

    files = sorted(set(files))

    if not files:
        searched_paths = "\n".join(f"  - {path}" for path in output_paths)

        raise FileNotFoundError(
            f"No files found for variable {variable!r}.\n"
            f"Searched directories:\n{searched_paths}"
        )

    if start_ym is None and end_ym is None:
        return files

    if start_ym is None or end_ym is None:
        raise ValueError(
            "start_ym and end_ym must either both be supplied "
            "or both be omitted."
        )

    if month_index(start_ym) > month_index(end_ym):
        raise ValueError(
            f"start_ym {start_ym} occurs after end_ym {end_ym}"
        )

    candidates = []

    for path in files:
        parsed = parse_model_filename(path)

        if parsed is None:
            continue

        if parsed["variable"] != variable:
            continue

        if (
            month_index(parsed["end_ym"]) >= month_index(start_ym)
            and month_index(parsed["start_ym"]) <= month_index(end_ym)
        ):
            candidates.append(parsed)

    if not candidates:
        raise FileNotFoundError(
            f"No {variable!r} files overlap the requested period "
            f"{start_ym:06d}-{end_ym:06d}"
        )

    selected = select_continuous_files(
        candidates=candidates,
        start_ym=start_ym,
        end_ym=end_ym,
    )

    return [item["path"] for item in selected]


# ============================================================
# Model-selection helper
# ============================================================

def select_model_cases(
    model_version=None,
    configuration=None,
    experiment=None,
    member=None,
    variant=None,
):
    """
    Select model cases using configuration metadata.

    Each argument is optional. Supplied arguments are combined using
    logical AND.
    """
    selected = []

    for case_key, case in MODEL_CASES.items():
        if (
            model_version is not None
            and case["model_version"] != model_version
        ):
            continue

        if (
            configuration is not None
            and case["configuration"] != configuration
        ):
            continue

        if (
            experiment is not None
            and case["experiment"] != experiment
        ):
            continue

        if member is not None and case["member"] != member:
            continue

        if variant is not None and case["variant"] != variant:
            continue

        selected.append(case_key)

    return selected


# ============================================================
# Example model and period selection
# ============================================================

# Available model-suite presets are defined in util/asl_workflow.py.
# Use "v2_1_SORRM" to preserve the original SORRM historical+SSP370 workflow.
# Use "v2_1_LR" for the LR historical workflow from CVDP_RGD files.
# Use "cmip6_historical" or "cmip6_amip" for CMIP6 processing.
# Set cmip6_member_selection="r1i1p1f1" for first historical members, or "all" for all available members.
active_model_suite = "cmip6_historical"
cmip6_member_selection = "r1i1p1f1"

process_observations = False
# Existing raw indices are skipped unless force_recompute=True.
process_model_indices = True
process_control_index = False
process_climatology = True
# Lead-lag is expensive for CMIP6; enable after raw indices and climatology are ready.
process_lead_lag = False
force_recompute = False

suite_config = asl_workflow.load_suite_config(
    active_model_suite,
    select_model_cases,
    cmip6_member_selection=cmip6_member_selection,
)
selected_cases = suite_config["selected_cases"]
analysis_output = suite_config["analysis_output"]
analysis_variable = suite_config["analysis_variable"]
asl_version = "asl_scotthoskingv3"
workflow_periods = suite_config["workflow_periods"]
preview_periods = list(workflow_periods)
preview_case_limit = suite_config.get("preview_case_limit")

# ============================================================
# Workflow configuration
# ============================================================

diagnostic_data_root = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/data/asl_analysis"
diagnostic_figure_root = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/figure/asl_analysis"

raw_index_path = os.path.join(diagnostic_data_root, "raw_index")
ts_index_path = os.path.join(diagnostic_data_root, "ts_index")
clim_index_path = os.path.join(diagnostic_data_root, "clim_index")
lead_lag_out = os.path.join(diagnostic_data_root, "lead_lag")

index_fig_path = os.path.join(diagnostic_figure_root, "asl_index_ts")
lead_lag_fig = os.path.join(diagnostic_figure_root, "lead_lag")

input_source = suite_config.get("input_source", "pcmdi" if suite_config["use_pcmdi_model_input"] else "post")
use_pcmdi_model_input = input_source == "pcmdi"

control_case_key = suite_config["control_case_key"]
model_mip = suite_config.get("model_mip", "e3sm")
analysis_mip = "analysis"
control_experiment = suite_config["control_experiment"]
control_member = suite_config["control_member"]
control_period = suite_config["control_period"]

obs_names = asl_workflow.OBS_CONFIG["names"]
obs_experiment = asl_workflow.OBS_CONFIG["experiment"]
obs_member = asl_workflow.OBS_CONFIG["member"]
obs_period_by_name = asl_workflow.OBS_CONFIG["period_by_name"]
obs_variable_by_name = asl_workflow.OBS_CONFIG["variable_by_name"]

clim_mips = suite_config["clim_mips"]
clim_exps = suite_config["clim_exps"]
clim_seasons_step1 = ["ANN", "DJF", "JJA", "MAM", "SON", "AC", "Monthly"]
clim_indices = ["lon", "lat", "ActCenPres", "SectorPres", "RelCenPres"]
clim_obs_sets = asl_workflow.OBS_CONFIG["clim_sets"]
clim_obs_mips = asl_workflow.OBS_CONFIG["clim_mips"]
clim_obs_periods = asl_workflow.OBS_CONFIG["clim_periods"]
clim_test_mips = suite_config["clim_test_mips"]
clim_test_exps = suite_config["clim_test_exps"]
clim_seasons_step2 = ["ANN", "DJF", "JJA", "MAM", "SON", "AC"]

lead_lag_period = suite_config.get("lead_lag_period", workflow_periods[0])
lead_lag_index = "RelCenPres"

def get_case_config(case_key):
    suite_cases = suite_config.get("case_metadata_by_case", {})
    if case_key in suite_cases:
        return suite_cases[case_key]
    suite_control_cases = suite_config.get("control_case_metadata_by_case", {})
    if case_key in suite_control_cases:
        return suite_control_cases[case_key]
    return MODEL_CASES[case_key]


def get_case_member(case_key):
    if case_key == control_case_key:
        return control_member
    member_aliases = suite_config.get("member_alias_by_case", {})
    if case_key in member_aliases:
        return member_aliases[case_key]
    return get_case_config(case_key)["member"]


def get_analysis_experiment(case_key, period_key, start_ym, end_ym):
    experiment_by_period = suite_config.get("experiment_by_period", {})
    if period_key in experiment_by_period:
        return experiment_by_period[period_key]

    case = get_case_config(case_key)
    if case.get("experiment") == "historical+ssp245":
        if end_ym <= ANALYSIS_PERIODS["historical"]["end_ym"]:
            return "historical"
        if start_ym >= ANALYSIS_PERIODS["ssp245"]["start_ym"]:
            return "ssp245"
        return "historical+ssp245"

    if case_key not in MODEL_CASES:
        return case["experiment"]

    return identify_analysis_experiment(case_key, start_ym, end_ym)


def filter_files_by_period(files, start_ym=None, end_ym=None):
    if start_ym is None or end_ym is None:
        return files

    period_filtered_files = []
    for path in files:
        match = re.search(r"(\d{6})-(\d{6})\.nc$", os.path.basename(path))
        if match is None:
            period_filtered_files.append(path)
            continue

        file_start_ym = int(match.group(1))
        file_end_ym = int(match.group(2))
        if file_start_ym <= end_ym and file_end_ym >= start_ym:
            period_filtered_files.append(path)
    return period_filtered_files


def get_cvdp_rgd_model_files(case_key, experiment, variable, start_ym=None, end_ym=None):
    member = get_case_member(case_key) if case_key != control_case_key else control_member
    case = get_case_config(case_key)
    cvdp_root = suite_config["cvdp_rgd_root"]
    exp_dir = suite_config["cvdp_rgd_experiment_dirs"][experiment]
    file_prefix = suite_config.get("cvdp_rgd_file_prefix_by_experiment", {}).get(experiment, exp_dir)
    search_dir = os.path.join(cvdp_root, exp_dir)
    subdir_template = suite_config.get("cvdp_rgd_model_subdir_template")
    if subdir_template:
        search_dir = os.path.join(search_dir, subdir_template.format(case_name=case["case_name"], member=member, experiment=experiment))
    filename_template = suite_config.get("cvdp_rgd_file_pattern", "{file_prefix}.{member}.{variable}.*.nc")
    filename_pattern = filename_template.format(
        file_prefix=file_prefix,
        case_name=case["case_name"],
        member=member,
        experiment=experiment,
        variable=variable.upper(),
    )
    pattern = os.path.join(search_dir, filename_pattern)
    files = filter_files_by_period(sorted(glob.glob(pattern)), start_ym=start_ym, end_ym=end_ym)
    if not files:
        raise FileNotFoundError(f"No CVDP_RGD files matched: {pattern}")
    return files


def get_pcmdi_model_files(case_key, experiment, variable, start_ym=None, end_ym=None):
    case = MODEL_CASES[case_key]
    member = case["member"] if case["member"] is not None else control_member
    pattern = os.path.join(
        run_path,
        model_mip,
        experiment,
        "Amon",
        variable,
        f"{model_mip}.{experiment}.{case['case_name']}.{member}.mon.{variable}.*.nc",
    )
    files = filter_files_by_period(sorted(glob.glob(pattern)), start_ym=start_ym, end_ym=end_ym)

    if not files:
        raise FileNotFoundError(f"No PCMDI files matched: {pattern}")
    return files


def get_analysis_input_files(case_key, experiment, variable, start_ym, end_ym):
    if input_source == "cvdp_rgd":
        return get_cvdp_rgd_model_files(case_key, experiment, variable, start_ym=start_ym, end_ym=end_ym)

    if input_source == "pcmdi":
        return get_pcmdi_model_files(case_key, experiment, variable, start_ym=start_ym, end_ym=end_ym)

    return get_model_files(
        case_key=case_key,
        output_key=analysis_output,
        variable=variable,
        start_ym=start_ym,
        end_ym=end_ym,
    )


def get_model_mask_file(experiment, case_name):
    if input_source in ("pcmdi", "cvdp_rgd"):
        return os.path.join(run_mask, "Amon", f"{model_mip}.{experiment}.{case_name}.fx.sftlf.nc")

    return os.path.join(run_mask, f"Amon/{model_mip}.{experiment}.{case_name}.fx.sftlf.nc")


preview_files_by_period_and_case = collections.OrderedDict()

for preview_period in preview_periods:
    preview_period_config = ANALYSIS_PERIODS[preview_period]
    preview_start_ym = preview_period_config["start_ym"]
    preview_end_ym = preview_period_config["end_ym"]
    preview_files_by_period_and_case[preview_period] = collections.OrderedDict()

    for case_key in selected_cases:
        preview_experiment = get_analysis_experiment(
            case_key,
            preview_period,
            start_ym=preview_start_ym,
            end_ym=preview_end_ym,
        )
        preview_files_by_period_and_case[preview_period][case_key] = {
            "experiment": preview_experiment,
            "files": get_analysis_input_files(
                case_key=case_key,
                experiment=preview_experiment,
                variable=analysis_variable,
                start_ym=preview_start_ym,
                end_ym=preview_end_ym,
            ),
        }

model_files = preview_files_by_period_and_case

asl_workflow.print_workflow_switches(
    active_model_suite,
    process_observations,
    process_model_indices,
    process_control_index,
    process_climatology,
    process_lead_lag,
    force_recompute,
)
print("")

print(f"Selected cases      : {len(selected_cases)}")
if model_mip == "cmip6":
    print(f"CMIP6 member mode   : {cmip6_member_selection}")
print(f"Preview periods     : {', '.join(preview_periods)}")
print(f"Analysis output     : {analysis_output}")
print(f"Analysis variable   : {analysis_variable}")
print("")

for preview_period, files_by_case in preview_files_by_period_and_case.items():
    preview_period_config = ANALYSIS_PERIODS[preview_period]
    preview_start_ym = preview_period_config["start_ym"]
    preview_end_ym = preview_period_config["end_ym"]
    print(f"Preview period      : {preview_period} ({preview_start_ym:06d}-{preview_end_ym:06d})")

    for preview_index, (case_key, preview_info) in enumerate(files_by_case.items()):
        if preview_case_limit is not None and preview_index >= preview_case_limit:
            remaining_cases = len(files_by_case) - preview_case_limit
            print(f"... {remaining_cases} additional cases validated but not printed")
            print("")
            break
        case = get_case_config(case_key)
        case_files = preview_info["files"]
        print(f"Selected case       : {case_key}")
        print(f"Simulation directory: {case['model_name']}")
        print(f"Selected experiment : {preview_info['experiment']}")
        print(f"Member              : {get_case_member(case_key)}")
        print(f"Number of files     : {len(case_files)}")

        for path in case_files:
            print(f"  {path}")

        print("")


# ============================================================
# Amundsen Sea Low configuration
# ============================================================

asl_region = {
    "west": 170.0,
    "east": 298.0,
    "south": -80.0,
    "north": -60.0,
}

asl_min_dist = 5
asl_num_peak = 3
asl_start_month = 1
asl_end_month = 12
asl_exc_bord = False
l_check_asl_region = False
l_allow_no_asl = False

Active model suite      : cmip6_historical
Process observations    : False
Process model indices   : True
Process control index   : False
Process climatology     : True
Process lead-lag        : False
Force recompute         : False

Selected cases      : 46
CMIP6 member mode   : r1i1p1f1
Preview periods     : historical
Analysis output     : atm_ts
Analysis variable   : PSL

Preview period      : historical (195101-201412)
Selected case       : CMIP6-historical-AS-RCEC_TaiESM1-r1i1p1f1_gn_v20200623
Simulation directory: AS-RCEC_TaiESM1.historical.r1i1p1f1_gn_v20200623
Selected experiment : historical
Member              : r1i1p1f1_gn_v20200623
Number of files     : 1
  /lcrc/group/e3sm/ac.szhang/acme_scratch/data/CVDP_RGD/CMIP6_MME_LTM/historical/AS-RCEC_TaiESM1/AS-RCEC_TaiESM1.historical.r1i1p1f1_gn_v20200623.PSL.185001-201412.nc

Selected case       : CMIP6-historical-AWI_AWI-CM-1-1-MR-r1i1p1f1_gn_v20181218
Simulation directory: AWI_AWI-CM-1-1-MR.historical.r1i1p1f1_gn_v20181218
Sel

## 1. Run ASL Index Generation

In [ ]:
# Define out and figure directories
out_path = raw_index_path
fig_path = index_fig_path

def format_ym_span(start_ym, end_ym):
    return f"{start_ym:06d}-{end_ym:06d}"


def run_model_asl_index_generation(period_key):
    period_config = ANALYSIS_PERIODS[period_key]
    period_start_ym = period_config["start_ym"]
    period_end_ym = period_config["end_ym"]
    period = format_ym_span(period_start_ym, period_end_ym)

    for case_key in selected_cases:
        case = get_case_config(case_key)
        member = get_case_member(case_key)
        experiment = get_analysis_experiment(
            case_key,
            period_key,
            start_ym=period_start_ym,
            end_ym=period_end_ym,
        )
        case_files = get_analysis_input_files(
            case_key=case_key,
            experiment=experiment,
            variable=analysis_variable,
            start_ym=period_start_ym,
            end_ym=period_end_ym,
        )
        case_dict = collections.OrderedDict()
        case_dict[case["case_name"]] = Case(case_files, analysis_variable, "blue", case["case_name"])

        expected_out = asl_workflow.raw_index_file(
            out_path,
            model_mip,
            experiment,
            case["case_name"],
            member,
            asl_version,
            analysis_variable,
            period,
        )
        if not asl_workflow.should_run(expected_out, force_recompute, "model raw ASL index"):
            continue

        mask_file = get_model_mask_file(experiment, case["case_name"])
        if os.path.exists(mask_file):
            case_dict["mask"] = Case(mask_file, "sftlf", "blue", case["case_name"])

        run_asl_index_generation(
            fig_path,
            out_path,
            model_mip,
            experiment,
            member,
            asl_version,
            period,
            case_dict,
            asl_region,
            asl_start_month,
            asl_end_month,
            asl_exc_bord,
            l_check_asl_region,
            l_allow_no_asl,
        )


# 1.1 E3SM model members
if process_model_indices:
    for workflow_period in workflow_periods:
        run_model_asl_index_generation(workflow_period)
else:
    print("Skip model ASL index generation: process_model_indices=False")

# 1.2 Control E3SM run
if process_control_index and control_case_key is not None:
    case_dict = collections.OrderedDict()
    ctrl_case_key = control_case_key
    ctrl_case = get_case_config(ctrl_case_key)
    ctrl_period_config = ANALYSIS_PERIODS[control_period]
    control_period_span = asl_workflow.format_ym_span(ctrl_period_config["start_ym"], ctrl_period_config["end_ym"])
    expected_out = asl_workflow.raw_index_file(
        out_path,
        model_mip,
        control_experiment,
        ctrl_case["case_name"],
        control_member,
        asl_version,
        analysis_variable,
        control_period_span,
    )

    if asl_workflow.should_run(expected_out, force_recompute, "control raw ASL index"):
        control_files = get_analysis_input_files(
            case_key=ctrl_case_key,
            experiment=control_experiment,
            variable=analysis_variable,
            start_ym=ctrl_period_config["start_ym"],
            end_ym=ctrl_period_config["end_ym"],
        )
        case_dict[ctrl_case["case_name"]] = Case(control_files, analysis_variable, "blue", ctrl_case["case_name"])
        mask_file = get_model_mask_file(control_experiment, ctrl_case["case_name"])
        if os.path.exists(mask_file):
            case_dict['mask'] = Case(mask_file, "sftlf", "blue", ctrl_case["case_name"])
        run_asl_index_generation(fig_path, out_path, model_mip, control_experiment, control_member, asl_version, control_period_span,
                             case_dict, asl_region, asl_start_month, asl_end_month, asl_exc_bord, l_check_asl_region, l_allow_no_asl)
else:
    print("Skip control ASL index generation: process_control_index=False or no control case configured")

if process_control_index:
    for extra_control in suite_config.get("extra_control_runs", []):
        extra_case_key = extra_control["case_key"]
        extra_case = get_case_config(extra_case_key)
        extra_experiment = extra_control["experiment"]
        extra_member = extra_control["member"]
        extra_period_config = ANALYSIS_PERIODS[extra_control["period"]]
        extra_period_span = asl_workflow.format_ym_span(extra_period_config["start_ym"], extra_period_config["end_ym"])
        expected_out = asl_workflow.raw_index_file(
            out_path,
            model_mip,
            extra_experiment,
            extra_case["case_name"],
            extra_member,
            asl_version,
            analysis_variable,
            extra_period_span,
        )

        if not asl_workflow.should_run(expected_out, force_recompute, "extra control raw ASL index"):
            continue

        case_dict = collections.OrderedDict()
        control_files = get_analysis_input_files(
            case_key=extra_case_key,
            experiment=extra_experiment,
            variable=analysis_variable,
            start_ym=extra_period_config["start_ym"],
            end_ym=extra_period_config["end_ym"],
        )
        case_dict[extra_case["case_name"]] = Case(control_files, analysis_variable, "blue", extra_case["case_name"])
        mask_file = get_model_mask_file(extra_experiment, extra_case["case_name"])
        if os.path.exists(mask_file):
            case_dict['mask'] = Case(mask_file, "sftlf", "blue", extra_case["case_name"])
        run_asl_index_generation(fig_path, out_path, model_mip, extra_experiment, extra_member, asl_version, extra_period_span,
                                 case_dict, asl_region, asl_start_month, asl_end_month, asl_exc_bord, l_check_asl_region, l_allow_no_asl)

# 1.4 Observation/Analysis Runs (NOAA_20C and ERA5)
if process_observations:
    for obs_name in obs_names:
        obs_variable = obs_variable_by_name[obs_name]
        obs_period = obs_period_by_name[obs_name]
        expected_out = asl_workflow.raw_index_file(
            out_path,
            analysis_mip,
            obs_experiment,
            obs_name,
            obs_member,
            asl_version,
            obs_variable,
            obs_period,
        )
        if not asl_workflow.should_run(expected_out, force_recompute, "obs raw ASL index"):
            continue

        case_dict = collections.OrderedDict()
        obs_file = sorted(glob.glob(os.path.join(run_path, analysis_mip, obs_name, "Amon/psl/{}.{}.{}.{}.*.nc".format(analysis_mip, obs_experiment, obs_name, obs_member))))[0]
        case_dict[obs_name] = Case(obs_file, obs_variable, "blue", obs_name)
        mask_file = os.path.join(run_mask, "Amon/{}.{}.{}.fx.sftlf.nc".format(analysis_mip, obs_experiment, obs_name))
        case_dict['mask'] = Case(mask_file, "sftlf", "blue", obs_name)
        run_asl_index_generation(fig_path, out_path, analysis_mip, obs_experiment, obs_member, asl_version, obs_period,
                                 case_dict, asl_region, asl_start_month, asl_end_month, asl_exc_bord, l_check_asl_region, l_allow_no_asl)
else:
    print("Skip observation ASL index generation: process_observations=False")

Skip existing model raw ASL index: /lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/data/asl_analysis/raw_index/cmip6.historical.AS-RCEC_TaiESM1.r1i1p1f1_gn_v20200623.asl_scotthoskingv3.PSL.195101-201412.csv
Skip existing model raw ASL index: /lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/data/asl_analysis/raw_index/cmip6.historical.AWI_AWI-CM-1-1-MR.r1i1p1f1_gn_v20181218.asl_scotthoskingv3.PSL.195101-201412.csv
Skip existing model raw ASL index: /lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/data/asl_analysis/raw_index/cmip6.historical.AWI_AWI-ESM-1-1-LR.r1i1p1f1_gn_v20200212.asl_scotthoskingv3.PSL.195101-201412.csv
Skip existing model raw ASL index: /lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/data/asl_analysis/raw_index/cmip6.historical.BCC_BCC-CSM2-MR.r1i1p1f1_gn_v20181126.asl_scotthoskingv3.PSL.195101-201412.csv
working on ASL index generation for: BCC_BCC-ESM1 PSL
Time coordinate c

## 2. Run ASL Climatology Index Calculations (Step 1 & Step 2)

In [ ]:
# Directories for Climatology
# Paths and run lists are configured in the workflow setup cell.

if process_climatology:
    # Step 1: Calculate Seasonal/Annual Cycle
    run_asl_index_clim_step1(
        clim_mips,
        clim_exps,
        asl_version,
        clim_seasons_step1,
        clim_indices,
        raw_index_path,
        ts_index_path,
        force_recompute=force_recompute,
        print_skip=False,
    )

    # Step 2: Compare against Observation Climatologies
    run_asl_index_clim_step2(
        clim_obs_sets,
        clim_obs_mips,
        clim_test_mips,
        clim_test_exps,
        clim_obs_periods,
        clim_seasons_step2,
        clim_indices,
        ts_index_path,
        clim_index_path,
        force_recompute=force_recompute,
    )
else:
    print("Skip climatology processing: process_climatology=False")

## 3. Run ASL Lead-Lag Analysis

In [ ]:
# Directories for Lead-Lag
# Paths and run lists are configured in the workflow setup cell.

# E3SM historical model members
if process_lead_lag:
    lead_lag_period_config = ANALYSIS_PERIODS[lead_lag_period]
    lead_lag_period_span = format_ym_span(
        lead_lag_period_config["start_ym"],
        lead_lag_period_config["end_ym"],
    )

    for case_key in selected_cases:
        case = get_case_config(case_key)
        member = get_case_member(case_key)
        experiment = get_analysis_experiment(
            case_key,
            lead_lag_period,
            start_ym=lead_lag_period_config["start_ym"],
            end_ym=lead_lag_period_config["end_ym"],
        )
        expected_out = asl_workflow.lead_lag_file(
            lead_lag_out,
            lead_lag_index,
            model_mip,
            experiment,
            case["case_name"],
            member,
            asl_version,
            analysis_variable,
            lead_lag_period_span,
        )
        if not asl_workflow.should_run(expected_out, force_recompute, "lead-lag output"):
            continue

        case_files = get_analysis_input_files(
            case_key=case_key,
            experiment=experiment,
            variable=analysis_variable,
            start_ym=lead_lag_period_config["start_ym"],
            end_ym=lead_lag_period_config["end_ym"],
        )
        case_dict = collections.OrderedDict()
        case_dict[case["case_name"]] = Case(case_files, analysis_variable, "blue", case["case_name"])
        run_asl_leadlag_analysis(
            lead_lag_fig,
            lead_lag_out,
            model_mip,
            experiment,
            member,
            asl_version,
            lead_lag_period_span,
            case_dict,
            asl_region,
            lead_lag_index,
            l_check_asl_region,
        )
else:
    print("Skip lead-lag processing: process_lead_lag=False")